## Merge Model Dataset

Extracts pre- and peri-event waveform features (HR, SpO2) for every arrhythmia episode,
merges all data sources into a single modeling-ready file, then joins the outcome label:

1. `arrhythmia_episodes_updated.csv` — episode features (base)
2. `clinical_lab_updated.csv` — patient-level clinical and lab values
3. Waveform features extracted live from VitalDB (HR + SpO2)
4. `episode_hypotension_labels.csv` — joined in the final cell of this notebook

**Output:** `model_dataset.csv` — one row per episode (all 1,284), features + `hypotension_label`.
See the final cell for how episodes with no usable outcome window are handled.

In [ ]:
import pandas as pd
import numpy as np
import vitaldb
import os
import warnings
warnings.filterwarnings('ignore')

BASE_DIR       = os.path.abspath('..')
DATA_INTERIM   = os.path.join(BASE_DIR, 'data', 'interim')
DATA_PROCESSED = os.path.join(BASE_DIR, 'data', 'processed')
DATA_FINAL     = os.path.join(BASE_DIR, 'data', 'final')
os.makedirs(DATA_FINAL, exist_ok=True)

EPISODES_PATH = os.path.join(DATA_PROCESSED, 'arrhythmia_episodes_updated.csv')
CLINICAL_PATH = os.path.join(DATA_PROCESSED, 'clinical_lab_updated.csv')
IOH_PATH      = os.path.join(DATA_PROCESSED, 'episode_hypotension_labels.csv')
OUTPUT_PATH   = os.path.join(DATA_FINAL, 'model_dataset.csv')

HR_TRACK   = 'Solar8000/HR'
SPO2_TRACK = 'Solar8000/PLETH_SPO2'
INTERVAL   = 1.0

PRE_WINDOW     = 120
PERI_HALF      = 5
PRE_MIN_VALID  = 10
PERI_MIN_VALID = 3

# Physiological plausibility bounds — values outside these are monitor
# dropout or probe artifact and must be excluded before computing statistics.
# Mirrors the constants used in 09_ioh_labeling.ipynb.
HR_MIN,   HR_MAX   = 20, 250   # bpm  — HR=0 is dropout; >250 is artifact
SPO2_MIN, SPO2_MAX = 85, 100   # %    — <85 is probe disconnect/severe artifact

print('Paths and constants loaded.')
print(f'  Episodes:   {EPISODES_PATH}')
print(f'  Clinical:   {CLINICAL_PATH}')
print(f'  IOH labels: {IOH_PATH}')
print(f'  Output:     {OUTPUT_PATH}')
print(f'  HR range:   {HR_MIN}\u2013{HR_MAX} bpm')
print(f'  SpO2 range: {SPO2_MIN}\u2013{SPO2_MAX}%')

In [ ]:
def compute_pre_features(arr, start_sec, val_min, val_max):
    """
    120-second pre-event window: mean, slope, std.
    Removes NaN and out-of-range (artifact) values before computing.
    Returns (nan, nan, nan) if fewer than PRE_MIN_VALID clean readings remain.
    """
    s = max(0, int(start_sec) - PRE_WINDOW)
    e = int(start_sec)
    segment = arr[s:e].copy()
    segment[(segment < val_min) | (segment > val_max)] = np.nan
    valid_mask = ~np.isnan(segment)
    if valid_mask.sum() < PRE_MIN_VALID:
        return np.nan, np.nan, np.nan
    values = segment[valid_mask]
    t_idx  = np.where(valid_mask)[0]
    return float(np.mean(values)), float(np.polyfit(t_idx, values, 1)[0]), float(np.std(values))


def compute_peri_mean(arr, start_sec, val_min, val_max):
    """
    10-second peri-event window [start-5, start+5): mean.
    Removes NaN and out-of-range (artifact) values before computing.
    Returns NaN if fewer than PERI_MIN_VALID clean readings remain.
    """
    s = max(0, int(start_sec) - PERI_HALF)
    e = min(len(arr), int(start_sec) + PERI_HALF)
    segment = arr[s:e].copy()
    segment[(segment < val_min) | (segment > val_max)] = np.nan
    valid = segment[~np.isnan(segment)]
    return float(np.mean(valid)) if len(valid) >= PERI_MIN_VALID else np.nan


print('Helper functions defined (with artifact cleaning).')

In [ ]:
episodes = pd.read_csv(EPISODES_PATH)
caseids  = episodes['caseid'].unique()
n_total  = len(caseids)

print(f'Extracting waveform features for {len(episodes)} episodes across {n_total} patients.')
print('Loading from VitalDB API — this will take several minutes.\n')

feature_rows  = []
n_load_errors = 0

for i, caseid in enumerate(caseids):
    if i == 0 or (i + 1) % 50 == 0 or i == n_total - 1:
        print(f'  [{i+1}/{n_total}] caseid={caseid}')
    try:
        raw       = vitaldb.load_case(caseid, [HR_TRACK, SPO2_TRACK], INTERVAL)
        hr_full   = raw[:, 0]
        spo2_full = raw[:, 1]
    except Exception as e:
        print(f'    WARNING: load failed for caseid {caseid}: {e}')
        n_load_errors += 1
        hr_full   = np.array([np.nan])
        spo2_full = np.array([np.nan])

    for _, ep in episodes[episodes['caseid'] == caseid].iterrows():
        start = ep['episode_start_sec']
        # Pass explicit physiological bounds — artifact values (HR=0, SpO2<85) are
        # set to NaN inside the helper before statistics are computed.
        hr_pre_mean,   hr_pre_slope,   hr_pre_std   = compute_pre_features(
            hr_full,   start, HR_MIN,   HR_MAX)
        spo2_pre_mean, spo2_pre_slope, spo2_pre_std = compute_pre_features(
            spo2_full, start, SPO2_MIN, SPO2_MAX)
        hr_peri_mean   = compute_peri_mean(hr_full,   start, HR_MIN,   HR_MAX)
        spo2_peri_mean = compute_peri_mean(spo2_full, start, SPO2_MIN, SPO2_MAX)
        feature_rows.append({
            'caseid':          int(caseid),
            'episode_number':  int(ep['episode_number']),
            'hr_pre_mean':     hr_pre_mean,   'hr_pre_slope':    hr_pre_slope,
            'hr_pre_std':      hr_pre_std,    'spo2_pre_mean':   spo2_pre_mean,
            'spo2_pre_slope':  spo2_pre_slope,'spo2_pre_std':    spo2_pre_std,
            'hr_peri_mean':    hr_peri_mean,  'spo2_peri_mean':  spo2_peri_mean,
        })

waveform_df = pd.DataFrame(feature_rows)
print(f'\nExtraction complete: {len(waveform_df)} rows x {len(waveform_df.columns)} columns.')
if n_load_errors > 0:
    print(f'WARNING: {n_load_errors} patient(s) had load errors — waveform features are NaN.')

print('\nWaveform feature missingness (after artifact cleaning):')
for col in [c for c in waveform_df.columns if c not in ['caseid', 'episode_number']]:
    pct = waveform_df[col].isna().mean() * 100
    print(f'  {col}: {pct:.1f}% missing')

In [ ]:
episodes = pd.read_csv(EPISODES_PATH)
clinical  = pd.read_csv(CLINICAL_PATH)

print('Source shapes before merge:')
print(f'  episodes:    {episodes.shape}')
print(f'  clinical:    {clinical.shape}')
print(f'  waveform_df: {waveform_df.shape}')

clinical_clean = clinical.drop(columns=['bp_source'], errors='ignore')
df = episodes.merge(clinical_clean, on='caseid', how='left')
print(f'\nAfter episodes + clinical:  {df.shape}')
if len(df) != len(episodes):
    print('  WARNING: row count changed — investigate!')

waveform_df['caseid']         = waveform_df['caseid'].astype(int)
waveform_df['episode_number'] = waveform_df['episode_number'].astype(int)
df['caseid']                  = df['caseid'].astype(int)
df['episode_number']          = df['episode_number'].astype(int)

df = df.merge(waveform_df, on=['caseid', 'episode_number'], how='left')
print(f'After + waveform features:  {df.shape}')
if len(df) != len(episodes):
    print('  WARNING: row count changed — investigate!')

print(f'\nFinal feature dataset shape: {df.shape}')
print('Join to IOH_labels/episode_hypotension_labels.csv at training time for labels.')

In [ ]:
print('=== CHECK 1: Row count ===')
print(f'Rows: {len(df)}  (expected 1,284)')
if len(df) != 1284:
    print('WARNING: unexpected row count')
else:
    print('OK')

print('\n=== CHECK 2: Key identifiers ===')
print('OK' if df['caseid'].isna().sum() == 0 and df['episode_number'].isna().sum() == 0
      else 'PROBLEM \u2014 null identifiers')

print('\n=== CHECK 3: No outcome columns leaked in from the merge sources ===')
outcome_cols = ['hypotension_label','bp_outcome_min','bp_outcome_mean','outcome_window_availability']
leaked = [c for c in outcome_cols if c in df.columns]
if leaked:
    df = df.drop(columns=leaked)
    print(f'Removed leaked columns: {leaked}')
else:
    print('OK \u2014 no outcome columns present yet (joined explicitly in the next cell)')

print('\n=== CHECK 4: Missingness ===')
miss = (df.isna().mean() * 100).sort_values(ascending=False)
for col, pct in miss[miss > 0].items():
    print(f'  {col}: {pct:.1f}%')
print(f'{int((miss == 0).sum())} columns fully complete.')

print(f'\nFeature merge complete: {df.shape[0]} episodes x {df.shape[1]} columns (label joined next).')

In [ ]:
# Join the hypotension outcome label
# Done here (not deferred to training time) so model_dataset.csv is the single,
# complete, ready-to-train file.
#
# For ~88 episodes (6.9%), the post-episode monitoring window had essentially no
# BP readings (outcome_window_availability near 0) -- there's no way to confirm
# whether hypotension occurred. We recode these as hypotension_label=0 (no
# confirmed complication) rather than dropping them: inconclusive monitoring is
# not evidence of an adverse outcome. This is a deliberate methodological choice,
# not a default -- see DATA.md for the full rationale and the alternative
# (drop-NaN) that was considered.

ioh_labels = pd.read_csv(IOH_PATH)

df = df.merge(
    ioh_labels[['caseid', 'episode_number', 'hypotension_label']],
    on=['caseid', 'episode_number'], how='left'
)

n_unlabeled = int(df['hypotension_label'].isna().sum())
print(f'Episodes with no usable outcome window: {n_unlabeled} '
      f'({n_unlabeled / len(df) * 100:.1f}%) -- recoded to hypotension_label=0')
df['hypotension_label'] = df['hypotension_label'].fillna(0).astype(int)

print('\nFinal label distribution:')
print(df['hypotension_label'].value_counts().to_string())

df.to_csv(OUTPUT_PATH, index=False)
print(f'\nSaved -> {OUTPUT_PATH}')
print(f'model_dataset.csv: {df.shape[0]} episodes x {df.shape[1]} columns (includes hypotension_label)')
print(f'Unique patients:   {df["caseid"].nunique()}')